# GĐ2 — EDA tập IT đã làm sạch
Đọc `data/processed/it_salary_clean.csv` (tạo bởi `python src/data_cleaning.py`).
Số liệu in ra màn hình + `reports/gd2_eda_output.txt`; biểu đồ lưu ở `reports/figures/gd2_*.png`.

Lưu ý chống rò rỉ: EDA ở đây chỉ MÔ TẢ toàn bộ tập sạch. Mọi lựa chọn có "học" từ dữ liệu
(chọn top-k kỹ năng, gộp nhóm hiếm, impute) sẽ fit trên TRAIN ở GĐ3.
Không diễn giải chênh lệch lương giữa các nhóm là quan hệ nhân quả.

In [ ]:
import warnings
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)


def find_root() -> Path:
    start = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / "data" / "processed").is_dir():
            return p
    raise FileNotFoundError("Không tìm thấy data/processed/. Chạy `python src/data_cleaning.py` trước.")


ROOT = find_root()
PROCESSED = ROOT / "data" / "processed" / "it_salary_clean.csv"
INTERIM = ROOT / "data" / "interim" / "vietjobs_it_candidates.csv"
FIG_DIR = ROOT / "reports" / "figures"
OUT_TXT = ROOT / "reports" / "gd2_eda_output.txt"
FIG_DIR.mkdir(parents=True, exist_ok=True)
EXPERIENCE_ORDER = ["0 (không yêu cầu)", "<1 năm", "1 năm", "2 năm", "3 năm", "4+ năm", "unknown"]
MIN_GROUP_N = 30

_lines = []


def log(*parts):
    text = " ".join(str(p) for p in parts)
    print(text)
    _lines.append(text)


def section(title):
    log("")
    log("=" * 90)
    log(title)
    log("=" * 90)


def save(fig, name):
    path = FIG_DIR / f"gd2_{name}.png"
    fig.tight_layout()
    fig.savefig(path, dpi=120)
    log(f"[hình] {path.relative_to(ROOT)}")
    plt.show()
    plt.close(fig)

In [ ]:
df = pd.read_csv(PROCESSED, encoding="utf-8-sig")
cand = pd.read_csv(INTERIM, encoding="utf-8-sig")
section("1. KÍCH THƯỚC")
log(f"Tập sạch (dùng cho mô hình): {len(df):,} dòng, {df.shape[1]} cột")
log(f"Ứng viên IT (interim, gồm cả dòng bị loại): {len(cand):,} dòng")
log("Lý do loại trong các dòng is_it=True:")
log(cand.loc[cand["is_it"], "model_reason"].value_counts().to_string())

In [ ]:
section("2. TỶ LỆ THIẾU")
miss = pd.DataFrame({
    "pct_missing": (100 * df.isna().mean()).round(1),
    "n_unique": df.nunique(),
})
log(miss.to_string())
log(f"\nskills rỗng: {100 * (df['n_skills'] == 0).mean():.1f}% | "
    f"seniority 'unknown': {100 * (df['seniority'] == 'unknown').mean():.1f}% | "
    f"experience_group 'unknown': {100 * (df['experience_group'] == 'unknown').mean():.1f}%")
log("Ghi chú: cột `date` rỗng 100% vì VietJobs không có ngày đăng.")

In [ ]:
section("3. PHÂN BỐ NHÃN salary_label (triệu VND/tháng, trung điểm = giả định)")
y = df["salary_label"]
log(y.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).round(2).to_string())
log(f"Độ lệch (skew): {y.skew():.2f} | skew của log(y): {np.log(y).skew():.2f}")
q1, q3 = y.quantile([.25, .75])
upper = q3 + 1.5 * (q3 - q1)
out = df[y > upper]
log(f"Ngoại lệ theo IQR (> {upper:.1f}): {len(out):,} tin ({100 * len(out) / len(df):.1f}%) — CHỈ đánh dấu, không loại")
log("10 tin lương cao nhất:")
log(df.nlargest(10, "salary_label")[["job_title", "job_category", "experience_raw", "city", "salary_raw"]].to_string())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(y, bins=40, color="#4C72B0")
axes[0].set(title="salary_label", xlabel="triệu VND/tháng", ylabel="số tin")
axes[1].hist(np.log(y), bins=40, color="#55A868")
axes[1].set(title="log(salary_label)", xlabel="log(triệu)", ylabel="số tin")
save(fig, "label_hist")

In [ ]:
section("4. NHÃN THEO NHÓM (n, trung vị, Q1, Q3)")


def group_stats(col, order=None):
    g = df.groupby(col)["salary_label"].agg(
        n="size", median="median", q25=lambda s: s.quantile(.25), q75=lambda s: s.quantile(.75))
    g = g.reindex([o for o in order if o in g.index]) if order else g.sort_values("n", ascending=False)
    g["n<30"] = np.where(g["n"] < MIN_GROUP_N, "!", "")
    return g.round(1)


def box_by(col, name, order=None, max_groups=12):
    stats = group_stats(col, order)
    groups = list(stats.index)[:max_groups]
    data = [df.loc[df[col] == g, "salary_label"] for g in groups]
    fig, ax = plt.subplots(figsize=(10, 0.45 * len(groups) + 1.5))
    ax.boxplot(data, vert=False, showfliers=True)
    ax.set_yticks(range(1, len(groups) + 1))
    ax.set_yticklabels([f"{g} (n={stats.loc[g, 'n']})" for g in groups])
    ax.invert_yaxis()
    ax.set(xlabel="salary_label (triệu VND/tháng)", title=f"Lương đăng tuyển theo {col}")
    save(fig, name)


for col, order in [("experience_group", EXPERIENCE_ORDER), ("job_category", None), ("job_role", None),
                   ("city", None), ("seniority", None)]:
    log(f"\n--- {col}")
    log(group_stats(col, order).to_string())

box_by("experience_group", "box_experience", EXPERIENCE_ORDER)
box_by("job_category", "box_job_category")
box_by("city", "box_city")
box_by("seniority", "box_seniority")

In [ ]:
section("5. KINH NGHIỆM (số năm) và NHÃN — mô tả, không phải nhân quả")
rho = df[["experience_years", "salary_label"]].corr(method="spearman").iloc[0, 1]
log(f"Spearman(experience_years, salary_label) = {rho:.3f}")
fig, ax = plt.subplots(figsize=(7, 4))
jitter = np.random.default_rng(42).uniform(-0.15, 0.15, len(df))
ax.scatter(df["experience_years"] + jitter, df["salary_label"], s=8, alpha=0.4)
ax.set(xlabel="experience_years (có jitter)", ylabel="salary_label", title="Kinh nghiệm và lương đăng tuyển")
save(fig, "scatter_experience")

In [ ]:
section("6. ĐỘ RỘNG KHOẢNG LƯƠNG")
w = df["salary_width_ratio"]
log(w.describe(percentiles=[.5, .9, .95, .99]).round(2).to_string())
log(f"max/min > 2: {int((w > 2).sum()):,} tin | > 3: {int((w > 3).sum()):,} tin")

In [ ]:
section("7. KỸ NĂNG (đã chuẩn hóa đồng nghĩa) — chỉ thống kê tần suất")
skills = df["skills"].fillna("").map(lambda s: [x for x in s.split("|") if x])
counts = pd.Series([k for lst in skills for k in lst]).value_counts()
log(f"Số kỹ năng khác nhau: {len(counts):,}")
for k in (10, 20, 30, 50):
    log(f"   xuất hiện ≥ {k} tin: {int((counts >= k).sum()):,}")
for k in (20, 30, 50):
    top = set(counts.head(k).index)
    cov = skills.map(lambda lst: any(s in top for s in lst)).mean()
    log(f"   top-{k} kỹ năng phủ {100 * cov:.1f}% số tin (≥ 1 kỹ năng trong top)")
log("Top 40:")
log(counts.head(40).to_string())
fig, ax = plt.subplots(figsize=(8, 7))
counts.head(30)[::-1].plot.barh(ax=ax, color="#8172B2")
ax.set(xlabel="số tin", title="Top 30 kỹ năng (tập sạch)")
save(fig, "top_skills")

In [ ]:
section("8. TRÙNG MÔ TẢ (để chia train/test theo nhóm ở GĐ3)")
g = df["desc_group"].value_counts()
log(f"{int((g > 1).sum()):,} mô tả lặp lại ở {int(g[g > 1].sum()):,} tin; nhóm lớn nhất: {int(g.max())} tin")

In [ ]:
section("TÓM TẮT GĐ2")
log(f"Tập sạch: {len(df):,} tin | trung vị nhãn {y.median():.1f} triệu | IQR {q1:.1f}–{q3:.1f}")
for col in ["job_category", "city", "experience_group", "seniority"]:
    vc = df[col].value_counts()
    log(f"{col}: {len(vc)} nhóm, {int((vc < MIN_GROUP_N).sum())} nhóm có n < {MIN_GROUP_N}")
log(f"Kỹ năng xuất hiện ≥ 20 tin: {int((counts >= 20).sum())} | Spearman(kinh nghiệm, nhãn) = {rho:.3f}")
OUT_TXT.write_text("\n".join(_lines), encoding="utf-8")
print(f"\nĐã lưu: {OUT_TXT}")